In [5]:
import requests
import json
import os
import time
from datetime import datetime

# Task 1: Fetch Data from API - TrendPulse
# This script fetches top stories from HackerNews, categorizes them, and saves them into a structured JSON file for further processing.

def fetch_trendpulse_data():

    # API Endpoints
    TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
    ITEM_URL = "https://hacker-news.firebaseio.com/v0/item/{}.json"

    headers = {"User-Agent": "TrendPulse/1.0"}

    # Define categories and their respective keywords for matching

    categories = {
        "technology": ["AI", "software", "tech", "code", "computer", "data", "cloud", "API", "GPU", "LLM"],
        "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
        "sports": ["NFL", "NBA", "FIFA", "sport", "game", "team", "player", "league", "championship"],
        "science": ["research", "study", "space", "physics", "biology", "discovery", "NASA", "genome"],
        "entertainment": ["movie", "film", "music", "Netflix", "game", "book", "show", "award", "streaming"]
    }

    all_collected_stories = []
    category_counts = {cat: 0 for cat in categories}
    max_per_category = 25

    try:
        # Step 1: Get the list of top 500 story IDs
        response = requests.get(TOP_STORIES_URL, headers=headers)
        response.raise_for_status()
        story_ids = response.json()[:500]
    except Exception as e:
        print(f"Failed to fetch top stories: {e}")
        return

    print("Starting data collection...")

    # Step 2: Loop through categories to fetch and filter stories
    for category, keywords in categories.items():
        print(f"Processing category: {category}...")

        for story_id in story_ids:
            # Stop if we have enough stories for this category
            if category_counts[category] >= max_per_category:
                break

            try:
                # Fetch individual story details
                item_res = requests.get(ITEM_URL.format(story_id), headers=headers)
                item_res.raise_for_status()
                story = item_res.json()

                # Skip if story is deleted or doesn't have a title
                if not story or 'title' not in story:
                    continue

                title = story.get('title', '')
                title_lower = title.lower()

                # Check if any keyword matches the title (case-insensitive)
                if any(kw.lower() in title_lower for kw in keywords):
                    # Extract required fields
                    story_data = {
                        "post_id": story.get("id"),
                        "title": title,
                        "category": category,
                        "score": story.get("score", 0),
                        "num_comments": story.get("descendants", 0),
                        "author": story.get("by", "unknown"),
                        "collected_at": datetime.now().isoformat()
                    }

                    all_collected_stories.append(story_data)
                    category_counts[category] += 1

            except Exception:
                # If a single request fails, print and move on per requirements
                print(f"Skipping story ID {story_id} due to fetch error.")
                continue

        # Wait 2 seconds between each category loop
        time.sleep(2)

    # Step 3: Save to JSON file
    if not os.path.exists('data'):
        os.makedirs('data')

    timestamp = datetime.now().strftime("%Y%m%d")
    filename = f"data/trends_{timestamp}.json"

    with open(filename, 'w') as f:
        json.dump(all_collected_stories, f, indent=4)

    print(f"\nSuccess! Collected a total of {len(all_collected_stories)} stories.")
    for cat, count in category_counts.items():
        print(f"- {cat}: {count}")
    print(f"Data saved to {filename}")

if __name__ == "__main__":
    fetch_trendpulse_data()

Starting data collection...
Processing category: technology...
Processing category: worldnews...
Processing category: sports...
Processing category: science...
Processing category: entertainment...

Success! Collected a total of 104 stories.
- technology: 25
- worldnews: 25
- sports: 17
- science: 12
- entertainment: 25
Data saved to data/trends_20260410.json
